# PolySurrogate — Sample Workflow

This notebook demonstrates the **complete PolySurrogate pipeline** end to end,
using a small, fast example:

1. **Imports and configuration**
2. **Data generation** — random graphs → quantum amplitude vectors
3. **Surrogate-model training** — a 4-photon PNN
4. **Inverse-design demonstration** — recover a graph for a target quantum state

The pipeline is three independent, reusable scripts at the project root
(`data_generate.py`, `ml_model.py`, `inverse_design.py`), connected only
through the files they read/write (a dataset `.npz`, a model `.msgpack`) —
never through direct Python imports between them. This notebook imports each
one as a module and calls its existing functions directly; it does not
reimplement any of their logic.

**This is a small demo, not a research run.** Settings here (5,000 samples,
20 training epochs) are deliberately tiny so the whole notebook finishes in
well under a minute. The surrogate will not be fully converged, so the
inverse-design fidelity at the end will be modest — that is expected. The
goal is only to show how the pieces fit together; see the production
scripts/`PARAMETERS.md` for realistic settings (millions of samples, tens of
thousands of epochs).

## 1. Imports and configuration

In [1]:
import sys
from pathlib import Path

# Add the project root to sys.path so the project's own modules
# (data_generate, ml_model, inverse_design) can be imported directly,
# whether this notebook is launched from the project root or from inside
# notebooks/.
_cwd = Path.cwd()
REPO_ROOT = _cwd if (_cwd / "ml_model.py").exists() else _cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import time

import numpy as np
import jax

import data_generate
import ml_model
import inverse_design

print("Project root:", REPO_ROOT)
print("JAX backend :", jax.default_backend(), "-", jax.devices())

Project root: /home/bo48god/projects/deep_dreaming/PolySurrogate


ERROR:2026-07-31 15:39:26,355:jax._src.xla_bridge:475: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/bo48god/projects/deep_dreaming/env_jax/lib/python3.12/site-packages/jax/_src/xla_bridge.py", line 473, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/bo48god/projects/deep_dreaming/env_jax/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/bo48god/projects/deep_dreaming/env_jax/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to load the CU

JAX backend : cpu - [CpuDevice(id=0)]


**Configuration** — every setting used below lives in this one cell, and a
single fixed `SEED` drives every stage (data generation, model
initialisation/shuffling, and inverse-design optimisation), so the whole
notebook is reproducible run to run.

In [2]:
# ---------------------------------------------------------------------------
# Configuration -- edit these to experiment. Everything else in the pipeline
# uses the same defaults as the standalone scripts (data_generate.py,
# ml_model.py, inverse_design.py).
# ---------------------------------------------------------------------------
SEED = 42            # single fixed seed used throughout -> reproducible demo

NODES = 4             # photons / graph vertices
DIMENSIONS = 2        # local Hilbert-space dimension (2 = qubit-like)

N_SAMPLES = 5_000     # Section 2: dataset size
BATCH_SIZE = 200      # Section 3: must evenly divide N_SAMPLES * train_split (0.8)
NUM_EPOCHS = 20       # Section 3: training epochs
HIDDEN_DIM = 400      # Section 3: PNN hidden-layer size

OPT_STEPS = 300       # Section 4: inverse-design optimisation steps

# All demo output (dataset, trained model, inverse-design results) lives
# under this one folder, relative to the project root.
RESULTS_ROOT = REPO_ROOT / "results" / "sample_workflow"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("Demo output will be written under:", RESULTS_ROOT.relative_to(REPO_ROOT))

Demo output will be written under: results/sample_workflow


## 2. Data generation

The surrogate model needs training pairs of *(graph edge weights) →
(quantum amplitude vector)*. `data_generate.py`'s `generate_dataset()`:

1. Builds the PyTheus perfect-matching catalogue for an `n`-node graph.
2. Draws random graph weight vectors uniformly from `[-1, 1]`.
3. Computes the exact (unnormalised) quantum amplitude vector for each
   weight vector, using PyTheus.
4. Saves the result to disk as `dataset_merged.npz`.

We generate **5,000 samples** here — small enough to run in a couple of
seconds, versus the millions used for real training runs.

In [3]:
weights, amps, data_dir = data_generate.generate_dataset(
    vertices=NODES, dimensions=DIMENSIONS, n_samples=N_SAMPLES,
    batch_size=500, seed=SEED, shard_size=N_SAMPLES,
    normed_data=False, save_data=True,
    data_root=str(RESULTS_ROOT / "data_generation"),
)
DATASET_PATH = data_dir / "dataset_merged.npz"

print(f"Generated {weights.shape[0]} samples -> {DATASET_PATH.relative_to(REPO_ROOT)}")
print(f"weights shape: {weights.shape}   amps shape: {amps.shape}")

2026-07-31 15:39:55,128 | INFO | Log file: /home/bo48god/projects/deep_dreaming/PolySurrogate/results/sample_workflow/data_generation/n4/n4_0/logs/data_generation_20260731_153955.log
2026-07-31 15:39:55,130 | INFO | === Random Dense Graph Dataset Generation ===
2026-07-31 15:39:55,132 | INFO | JAX devices: [CpuDevice(id=0)]
2026-07-31 15:39:55,135 | INFO | vertices=4, dimensions=2
2026-07-31 15:39:55,136 | INFO | n_samples=5,000, batch_size=500
2026-07-31 15:39:55,138 | INFO | seed=42
2026-07-31 15:39:55,140 | INFO | normed_data=False
2026-07-31 15:39:55,141 | INFO | precision=float32
2026-07-31 15:39:55,143 | INFO | save_data=True  out_dir=/home/bo48god/projects/deep_dreaming/PolySurrogate/results/sample_workflow/data_generation/n4/n4_0
2026-07-31 15:39:55,144 | INFO | Precomputing PyTheus perfect-matching catalog...
2026-07-31 15:39:55,153 | INFO | num_edges=24  n_kets=16  catalog time=0.01s
2026-07-31 15:39:55,430 | INFO | JIT warmup...
2026-07-31 15:39:55,767 | INFO | Generating da

## 3. Surrogate-model training

`ml_model.py` trains a neural network to approximate the graph-weights →
amplitude-vector mapping computed exactly (but expensively) by PyTheus.
Two architectures are available (`utils.py`):

- **PNN** (Polynomial Neural Network) — one hidden `Dense` layer followed by
  a monomial activation `x ** (nodes / 2)`, then a final `Dense` layer to
  the output dimension. This is the architecture used for the project's
  real surrogate models.
- **FNN** (standard feedforward network) — several `Dense` layers with GELU
  activations, for comparison.

For this demo we train a **4-photon PNN** on the 5,000 samples generated
above, for **20 epochs**.

`ml_model.py` is configured through module-level constants rather than a
function argument (see its own top-of-file `Parameters` section) — so we
set the handful that matter for this demo directly on the imported module,
then call `ml_model.train()` unchanged. Everything else (learning rate,
weight decay, precision, ...) keeps `ml_model.py`'s own defaults.

In [4]:
ml_model.NODES = NODES
ml_model.MODEL_NAME = "PNN"
ml_model.HIDDEN_DIM = HIDDEN_DIM
ml_model.DATA_PATH = str(DATASET_PATH)
ml_model.DATA_SIZE = None            # use all N_SAMPLES generated in Section 2
ml_model.SPLIT_MODE = "contiguous"
ml_model.BATCH_SIZE = BATCH_SIZE
ml_model.NUM_EPOCHS = NUM_EPOCHS
ml_model.PATIENCE = NUM_EPOCHS       # demo is short -- don't early-stop
ml_model.LR_DECAY_UNTIL_EPOCH = NUM_EPOCHS
ml_model.SEED = SEED
ml_model.ROOT_FOLDER = str(RESULTS_ROOT / "model_training")
ml_model.RESUME_RUN_DIR = None
ml_model.CONFIG.update(dict(
    nodes=ml_model.NODES, model_name=ml_model.MODEL_NAME, hidden_dim=ml_model.HIDDEN_DIM,
    data_path=ml_model.DATA_PATH, data_size=ml_model.DATA_SIZE, split_mode=ml_model.SPLIT_MODE,
    batch_size=ml_model.BATCH_SIZE, num_epochs=ml_model.NUM_EPOCHS, patience=ml_model.PATIENCE,
    lr_decay_until_epoch=ml_model.LR_DECAY_UNTIL_EPOCH, seed=ml_model.SEED,
))

run_dir = ml_model.train()
print(f"\nModel and logs saved in: {run_dir.relative_to(REPO_ROOT)}")

2026-07-31 15:40:03 | INFO | Run directory: /home/bo48god/projects/deep_dreaming/PolySurrogate/results/sample_workflow/model_training/PNN/n4/n4_0
2026-07-31 15:40:03 | INFO | Backend: cpu | devices: [CpuDevice(id=0)]
2026-07-31 15:40:03 | INFO | Precision=float32 | validation every epoch | plotting disabled


2026-07-31 15:40:06 | INFO | Setup seconds | hash=0.01 load=0.01 split=0.00 transfer=0.00 model_init=1.43 optimizer_init=0.13 jit_compile=1.03
2026-07-31 15:40:07 | INFO | epoch=0 | train=5.66096187e-01 | validation=5.06948650e-01 | shuffle=0.01s | train_time=0.15s | validation_time=0.01s | epoch_wall=0.94s | latest_ckpt=True | best_ckpt=True | best_epoch=0 | wait=0/20
2026-07-31 15:40:07 | INFO | epoch=1 | train=4.63985443e-01 | validation=4.26237762e-01 | shuffle=0.01s | train_time=0.02s | validation_time=0.00s | epoch_wall=0.45s | latest_ckpt=True | best_ckpt=True | best_epoch=1 | wait=0/20
2026-07-31 15:40:08 | INFO | epoch=2 | train=3.86559516e-01 | validation=3.61816227e-01 | shuffle=0.01s | train_time=0.02s | validation_time=0.00s | epoch_wall=0.40s | latest_ckpt=True | best_ckpt=True | best_epoch=2 | wait=0/20
2026-07-31 15:40:08 | INFO | epoch=3 | train=3.24885309e-01 | validation=3.06940377e-01 | shuffle=0.01s | train_time=0.02s | validation_time=0.00s | epoch_wall=0.39s | la

2026-07-31 15:40:15 | INFO | Complete | {'early_stopped': False, 'completed_epochs': 20, 'best_epoch': 19, 'best_validation_loss': 0.03788977488875389, 'dataset_source_dtypes': {'input_dtype': 'float32', 'target_dtype': 'float32'}, 'dataset_hash_seconds': 0.007153126876801252, 'dataset_load_seconds': 0.008445952087640762, 'dataset_split_seconds': 0.0008217771537601948, 'host_to_device_seconds': 0.001342426985502243, 'model_initialisation_seconds': 1.42934700101614, 'optimizer_initialisation_seconds': 0.1264703357592225, 'jit_compilation_seconds': 1.0295008230023086, 'training_loop_seconds': 8.504058211576194, 'total_shuffle_seconds': 0.19114536838606, 'total_train_seconds': 0.5097179873846471, 'total_validation_seconds': 0.038992826361209154, 'total_best_checkpoint_seconds': 3.6080760695040226, 'total_latest_checkpoint_seconds': 3.6525639765895903, 'mean_epoch_wall_seconds': 0.42176030210684984, 'best_restore_seconds': 0.0637193308211863, 'test_seconds': 0.37292668176814914, 'total_wal

Reading back the run's `summary.json` for a concise recap of the training result:

In [5]:
with open(run_dir / "summary.json") as f:
    summary = json.load(f)

print("Completed epochs     :", summary["completed_epochs"])
print("Best epoch            :", summary["best_epoch"])
print("Best validation loss  :", round(summary["best_validation_loss"], 6))
print("Test MAE               :", round(summary["test_mae"], 6))
print("Test fidelity (surrogate vs. exact PyTheus, on held-out samples):",
      round(summary["test_fidelity"], 6))

MODEL_PATH = run_dir / "best_params.msgpack"
assert MODEL_PATH.exists()

Completed epochs     : 20
Best epoch            : 19
Best validation loss  : 0.03789
Test MAE               : 0.03805
Test fidelity (surrogate vs. exact PyTheus, on held-out samples): 0.99057


## 4. Inverse-design demonstration

**Inverse design** goes the other direction: instead of predicting a
quantum state from a graph, we search for a graph whose predicted state
matches a chosen *target* state, by gradient descent on the graph weights
through the trained surrogate (used as a differentiable proxy for PyTheus).
`inverse_design.py` normally targets idealised states (GHZ, W, ...); here we
instead target **one specific sample's own quantum state**, generated back
in Section 2, so we can directly check how well the pipeline "inverts" a
known example.

> **Expect a modest fidelity here.** The surrogate above was trained on only
> 5,000 samples for 20 epochs — nowhere near enough to converge (a real run
> uses millions of samples and thousands of epochs). This section is only
> meant to demonstrate that the inverse-design machinery runs correctly
> end-to-end, not to produce a research-quality result.

In [6]:
# Pick one random sample from Section 2's dataset; its amplitude vector
# becomes the inverse-design target.
rng = np.random.default_rng(SEED)
sample_idx = int(rng.integers(0, N_SAMPLES))
target_state = amps[sample_idx].astype(np.float32)

print(f"Selected sample #{sample_idx} from the Section 2 dataset as the inverse-design target.")
print("Target amplitude vector:", np.round(target_state, 4))

Selected sample #446 from the Section 2 dataset as the inverse-design target.
Target amplitude vector: [ 0.212  -0.1966 -1.1393 -0.1858 -0.5105 -0.303   0.652  -0.1457 -0.316
 -0.4237 -1.0145 -0.1573 -1.2704 -0.7308  1.3067  0.1132]


`inverse_design.py`'s `get_target_state()` only knows about a fixed list of
named states (`GHZ`, `W`, `LINEAR_CLUSTER`, ...). To target our own sample's
state instead, we register one extra name, `"CUSTOM"`, that returns it —
everything else about the optimisation (`run_optimisation()`, jittering,
pruning, verification against exact PyTheus) is reused unchanged.

In [7]:
_original_get_target_state = inverse_design.get_target_state

def _get_target_state_or_custom(target_name, n):
    if target_name == "CUSTOM":
        return target_state
    return _original_get_target_state(target_name, n)

inverse_design.get_target_state = _get_target_state_or_custom

# Start from inverse_design.py's own defaults, then override only what
# this demo needs to change.
cfg = dict(inverse_design._default_cfg())
cfg.update(
    n=NODES, dimensions=DIMENSIONS, target_name="CUSTOM", model_type="PNN",
    generate_data=True, data_samples=1, data_batch_size=1, data_seed=SEED,
    normed_data=False, generation_gpu_batch_size=1, data_shard_size=1,
    architecture=HIDDEN_DIM, model_path=str(MODEL_PATH), normalize_model_output=False,
    input_dim=2 * NODES * (NODES - 1), out_dim=2 ** NODES,
    seed=SEED, num_steps=OPT_STEPS, max_total_steps=OPT_STEPS, lr_decay_steps=OPT_STEPS,
    results_root=str(RESULTS_ROOT / "inverse_design"),
)

result_dir = inverse_design.run_optimisation(cfg)
inverse_design.get_target_state = _original_get_target_state  # restore

print(f"\nInverse-design result saved in: {Path(result_dir).relative_to(REPO_ROOT)}")

2026-07-31 15:41:05,827 | INFO | === Random Dense Graph Dataset Generation ===
2026-07-31 15:41:05,830 | INFO | JAX devices: [CpuDevice(id=0)]
2026-07-31 15:41:05,831 | INFO | vertices=4, dimensions=2
2026-07-31 15:41:05,833 | INFO | n_samples=1, batch_size=1
2026-07-31 15:41:05,834 | INFO | seed=42
2026-07-31 15:41:05,836 | INFO | normed_data=False
2026-07-31 15:41:05,837 | INFO | precision=float32
2026-07-31 15:41:05,839 | INFO | save_data=False  out_dir=None
2026-07-31 15:41:05,841 | INFO | Precomputing PyTheus perfect-matching catalog...
2026-07-31 15:41:05,844 | INFO | num_edges=24  n_kets=16  catalog time=0.00s
2026-07-31 15:41:05,847 | INFO | JIT warmup...
2026-07-31 15:41:05,939 | INFO | Generating data batches...
2026-07-31 15:41:05,941 | INFO | batch     1 |          1/1 (100.0%) | 0.0s | 967 samples/s
2026-07-31 15:41:05,943 | INFO | === DONE === total time=0.0s
round=1 | generated=1 | kept=1 | discarded=0 | total_kept=1/1

Final low-fidelity dataset check:
total generated =

Comparing the target state to the optimised (PyTheus-verified) result:

In [8]:
with open(Path(result_dir) / "best_graph_solution.json") as f:
    best = json.load(f)

optimised_state = np.array(best["pytheus_state"])

print("Target state (Section 2, sample #{}):".format(sample_idx))
print(" ", np.round(target_state, 4))
print()
print("Optimised state found by inverse design (verified against exact PyTheus):")
print(" ", np.round(optimised_state, 4))
print()
print(f"PyTheus fidelity to target : {best['fidelity']:.4f}   (1.0 = perfect match)")
print(f"Active graph edges (after pruning): {best['num_edges']} / {2 * NODES * (NODES - 1)}")
print()
print("Note: this fidelity is modest because the surrogate above was trained")
print("on only 5,000 samples for 20 epochs -- this notebook demonstrates the")
print("workflow, not research-quality inverse design.")

Target state (Section 2, sample #446):
  [ 0.212  -0.1966 -1.1393 -0.1858 -0.5105 -0.303   0.652  -0.1457 -0.316
 -0.4237 -1.0145 -0.1573 -1.2704 -0.7308  1.3067  0.1132]

Optimised state found by inverse design (verified against exact PyTheus):
  [ 0.0675 -0.0347 -0.4292 -0.0657 -0.2064 -0.0652  0.2559 -0.0256 -0.0449
 -0.1621 -0.3732 -0.1253 -0.4475 -0.278   0.4809  0.026 ]

PyTheus fidelity to target : 0.9845   (1.0 = perfect match)
Active graph edges (after pruning): 24 / 24

Note: this fidelity is modest because the surrogate above was trained
on only 5,000 samples for 20 epochs -- this notebook demonstrates the
workflow, not research-quality inverse design.


---

### Summary

This notebook ran the full PolySurrogate pipeline on a tiny example:
generated 5,000 `(graph, quantum state)` pairs, trained a 4-photon PNN
surrogate for 20 epochs, and used that surrogate to invert one of the
generated samples back to a graph, reaching ~0.98 fidelity.

**For real results**, scale up the settings in the configuration cell
(Section 1) — production runs in this project use millions of samples,
thousands of training epochs, and tens of thousands of optimisation steps
(see the standalone scripts `data_generate.py`, `ml_model.py`,
`inverse_design.py`, and `PARAMETERS.md` for the full reference). Re-running
this notebook creates new auto-numbered output folders under
`results/sample_workflow/` each time, so nothing is overwritten.